## Concept 1 — ReAct Agent (Reason + Act)
First proper agent. Uses `create_react_agent` + `AgentExecutor`.

### vs Notebook 1 (LLM + Tools — Foundation)
```
LLM + Tools (Foundation — 1LLMWithTools.ipynb):
  YOUR code writes the loop manually.
  No reasoning trace — decisions are opaque.

ReAct Agent (this notebook):
  AgentExecutor manages the loop automatically.
  LLM writes a Thought before every Action — you can read WHY it chose each tool.
  You just call executor.invoke({'input': query}).
```

### What is ReAct?
ReAct = **Re**asoning + **Act**ing. Before every tool call the LLM writes a Thought.
This makes the agent's decision process fully transparent and debuggable.

### Pipeline
```
Query
  → Thought: what do I know? what do I need?
  → Action: tool_name
  → Action Input: args
  → Observation: tool result
  → [repeat until done]
  → Thought: I now know the final answer
  → Final Answer: ...
```

### Limitation (what Concept 2 fixes)
`AgentExecutor` + text-format parsing can be fragile if the LLM deviates from the format.
→ Concept 2 (Tool Calling Agent, `create_agent`) uses native tool calling — more robust.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from AgentUtils import get_llm, TOOLS, TEST_QUERIES, run_queries
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate

llm = get_llm()
print(f'LLM ready | Tools: {[t.name for t in TOOLS]}')

### Step 1 — The ReAct Prompt
The prompt instructs the LLM to follow the Thought/Action/Observation format exactly.
This format is what separates ReAct from plain tool calling.

In [ ]:
react_template = """You are a helpful assistant. Answer questions using the tools available.

Available tools:
{tools}

Use EXACTLY this format:

Question: the input question
Thought: think about what to do and why
Action: one of [{tool_names}]
Action Input: the input for the action
Observation: the result of the action
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I now know the final answer
Final Answer: the complete answer to the question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

react_prompt = PromptTemplate.from_template(react_template)
print('ReAct prompt template created')

### Step 2 — Create the ReAct Agent
`create_react_agent` wires LLM + tools + ReAct prompt.
`AgentExecutor` runs the Thought/Action/Observation loop until `Final Answer`.

In [ ]:
agent = create_react_agent(llm=llm, tools=TOOLS, prompt=react_prompt)

executor = AgentExecutor(
    agent=agent,
    tools=TOOLS,
    verbose=True,          # prints Thought/Action/Observation at each step
    max_iterations=6,      # safety limit to prevent infinite loops
    handle_parsing_errors=True,
)
print('ReAct agent ready')

### Step 3 — Watch the Thought/Action/Observation Loop
With `verbose=True` you can see every step of the agent's reasoning.

In [ ]:
result = executor.invoke({'input': 'What is 144 divided by 12?'})
print(f'\nFinal Answer: {result["output"]}')

### Step 4 — Multi-Step Reasoning
A query needing two tools shows how each Observation feeds into the next Thought.

In [ ]:
result = executor.invoke({'input': 'Search docs for RAG and also calculate 25 multiplied by 4'})
print(f'\nFinal Answer: {result["output"]}')

### Step 5 — No Tool Needed
ReAct agents can also answer directly when no tool is required.

In [ ]:
result = executor.invoke({'input': 'What does LCEL stand for in LangChain?'})
print(f'\nFinal Answer: {result["output"]}')

### Full Function

In [ ]:
def react_agent(query: str) -> str:
    result = executor.invoke({'input': query})
    return result['output']

### Test — Standard Queries

In [ ]:
run_queries(react_agent, label='Concept 2 — ReAct Agent')